# Notebook 01 — Data Loading & Cleaning
**Dataset:** IBM Telco Customer Churn  
**Tujuan:** Load CSV, bersihkan data, feature engineering, simpan ke Parquet.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path('../data/raw')
OUT = Path('../output')
OUT.mkdir(exist_ok=True)

## 1. Load Data

In [ ]:
df = pd.read_csv(RAW / 'WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
print(f'Churn rate: {df["Churn"].value_counts(normalize=True)["Yes"]:.1%}')
df.head(3)

## 2. Fix TotalCharges (string → numeric)

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_nan = df['TotalCharges'].isna().sum()
print(f'NaN setelah konversi: {n_nan} baris')

# Isi NaN dengan median (pelanggan tenure=0, baru bergabung)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print('TotalCharges dtype:', df['TotalCharges'].dtype)

## 3. Encode Target

In [ ]:
df['Churn'] = (df['Churn'] == 'Yes').astype(int)
print('Distribusi target:')
print(df['Churn'].value_counts())
print(f'Churn rate: {df["Churn"].mean():.1%}')

## 4. Feature Engineering

In [ ]:
# Segmen tenure
def tenure_segment(t):
    if t < 12:  return 'New (<12 bln)'
    elif t < 24: return 'Mid (12-24 bln)'
    else:        return 'Loyal (>24 bln)'

df['tenure_segment'] = df['tenure'].apply(tenure_segment)

# Jumlah layanan aktif
service_cols = ['PhoneService','MultipleLines','OnlineSecurity','OnlineBackup',
                'DeviceProtection','TechSupport','StreamingTV','StreamingMovies']
df['num_services'] = df[service_cols].apply(
    lambda row: sum(v == 'Yes' for v in row), axis=1
)

# Risk segment berdasarkan kontrak dan tenure
def risk_segment(row):
    if row['Contract'] == 'Month-to-month' and row['tenure'] < 12:
        return 'High Risk'
    elif row['Contract'] == 'Month-to-month':
        return 'Mid Risk'
    elif row['Contract'] == 'One year':
        return 'Mid Risk'
    else:
        return 'Low Risk'

df['risk_segment'] = df.apply(risk_segment, axis=1)

print('Kolom baru: tenure_segment, num_services, risk_segment')
print(df[['tenure_segment','num_services','risk_segment']].describe(include='all'))

## 5. Cek Kualitas Data

In [ ]:
print('Missing values:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'Tidak ada missing value.')

print('\nTipe data:')
print(df.dtypes)

## 6. Simpan ke Parquet

In [ ]:
df.to_parquet(OUT / 'df_clean.parquet', index=False)
print(f'Tersimpan: df_clean.parquet — {len(df):,} baris, {df.shape[1]} kolom')